# Notebook 02 — RLM Experiments

This notebook runs several experiments that highlight where Recursive Language
Models (RLMs) outperform plain single-shot LLM calls:

| Experiment | Concept demonstrated |
|---|---|
| 2-A | **Needle-in-a-Haystack** — find a hidden value in a long text |
| 2-B | **Hierarchical summarisation** — summarise multiple articles |
| 2-C | **Max depth & base case** — observe the fallback to plain LLM |
| 2-D | **Call-tree visualisation** — pretty-print the recursive structure |

## Setup

In [ ]:
import os
import sys
import json
import random
import textwrap

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

LLAMA_BASE_URL = os.environ.get('LLAMA_BASE_URL', 'http://host-gateway:8080/v1')
LLAMA_MODEL    = os.environ.get('LLAMA_MODEL',    'local-model')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'not-needed')

from rlm_smolagent import RLMAgent

def make_agent(max_depth=2, max_steps=10, verbose=False):
    return RLMAgent(
        base_url=LLAMA_BASE_URL,
        model_name=LLAMA_MODEL,
        api_key=OPENAI_API_KEY,
        max_depth=max_depth,
        max_steps=max_steps,
        verbose=verbose,
    )

print('Setup complete.')

## Helper: pretty-print the call tree

In [ ]:
def print_tree(node: dict, indent: int = 0) -> None:
    """Recursively print a call-tree dictionary produced by RLMAgent."""
    prefix = '  ' * indent
    depth  = node.get('depth', '?')
    dur    = node.get('duration_s', '?')
    prompt = node.get('prompt_preview', '')
    resp   = node.get('response_preview', '')
    print(f"{prefix}[depth={depth}] ({dur}s)")
    print(f"{prefix}  prompt  : {prompt}")
    print(f"{prefix}  response: {resp}")
    for child in node.get('children', []):
        print_tree(child, indent + 1)

---
## Experiment 2-A: Needle-in-a-Haystack

We hide a secret number inside a long passage of lorem-ipsum-like text and ask
the RLM to find it.  The RLM should split the haystack into chunks, search each
one recursively, and report the needle.

> This is the canonical "Hello World" example from the RLM paper.

In [ ]:
random.seed(7)

WORDS = (
    "the quick brown fox jumps over the lazy dog "
    "lorem ipsum dolor sit amet consectetur adipiscing elit "
).split()

def make_haystack(n_words: int = 800, needle_word: str = "SECRET_42") -> str:
    words = [random.choice(WORDS) for _ in range(n_words)]
    insert_pos = random.randint(n_words // 4, 3 * n_words // 4)
    words.insert(insert_pos, needle_word)
    return ' '.join(words)

needle  = "SECRET_42"
haystack = make_haystack(n_words=800, needle_word=needle)

print(f'Haystack length : {len(haystack.split())} words')
print(f'Needle          : "{needle}"')

In [ ]:
agent = make_agent(max_depth=2, max_steps=12, verbose=True)

prompt = textwrap.dedent(f"""\
    The text below contains the word \"{needle}\" exactly once.
    Your task: find that word and tell me the 5 words that appear immediately before it.

    Strategy:
      1. Split the text into two halves.
      2. Use rlm_call() to search each half.
      3. Return the context from whichever half found it.

    TEXT:
    {haystack}
""")

result = agent.completion(prompt)

print('\n=== RLM Answer ===')
print(result.response)

In [ ]:
print('\n=== Call Tree ===')
print_tree(result.metadata['call_tree'])

---
## Experiment 2-B: Hierarchical Summarisation

We generate three short "articles" and ask the RLM to produce:
1. A summary of each article (via recursive sub-calls).
2. A final meta-summary combining all three.

In [ ]:
ARTICLES = [
    (
        "Article A: Climate Change",
        "Global temperatures have risen by approximately 1.1°C above pre-industrial levels. "
        "Scientists warn that without drastic emission cuts, warming could exceed 2°C by 2100, "
        "leading to more frequent extreme weather events, rising sea levels, and biodiversity loss. "
        "Renewable energy adoption and policy changes are cited as the most critical levers."
    ),
    (
        "Article B: Artificial Intelligence",
        "Large language models (LLMs) have transformed natural language processing. "
        "Models with hundreds of billions of parameters can now write code, compose essays, "
        "and solve mathematical problems. Concerns around hallucination, bias, and energy "
        "consumption remain active research areas. Recursive inference techniques are emerging "
        "as a way to extend LLM capabilities beyond their context window."
    ),
    (
        "Article C: Space Exploration",
        "NASA's Artemis programme aims to return humans to the Moon by the late 2020s. "
        "SpaceX's Starship, the largest rocket ever built, is central to lunar and eventual "
        "Mars missions. Private companies have begun to lower the cost of orbital access "
        "significantly, opening new possibilities for satellite deployment and space tourism."
    ),
]

articles_text = '\n\n'.join(f'=== {title} ===\n{body}' for title, body in ARTICLES)
print(articles_text)

In [ ]:
agent = make_agent(max_depth=2, max_steps=12, verbose=True)

prompt = textwrap.dedent(f"""\
    You have three articles below.

    Step 1: Use rlm_call() three times — once per article — to get a
            one-sentence summary of each.
    Step 2: Combine the three summaries into a single 2-3 sentence
            meta-summary.

    Return only the final meta-summary.

    {articles_text}
""")

result = agent.completion(prompt)

print('\n=== Meta-Summary ===')
print(result.response)

In [ ]:
print('\n=== Call Tree ===')
print_tree(result.metadata['call_tree'])

---
## Experiment 2-C: Max Depth & Base Case

Set `max_depth=0` to force the agent to use the plain LLM fallback immediately.
This is useful for comparing the RLM answer vs. the naive single-shot answer.

In [ ]:
task = "Explain what a Recursive Language Model is in one paragraph."

# Plain single-shot (max_depth=0 → no recursion)
plain_agent = make_agent(max_depth=0, max_steps=5, verbose=False)
plain_result = plain_agent.completion(task)

# Full RLM (max_depth=2)
rlm_agent = make_agent(max_depth=2, max_steps=10, verbose=False)
rlm_result = rlm_agent.completion(task)

print('--- Plain (depth=0) ---')
print(plain_result.response)
print()
print('--- RLM (depth=2) ---')
print(rlm_result.response)

---
## Experiment 2-D: Saving and loading call-tree metadata

The `metadata` dict can be serialised to JSON for later analysis or
visualisation.

In [ ]:
import pathlib

log_dir = pathlib.Path('/workspace/logs')
log_dir.mkdir(parents=True, exist_ok=True)

log_file = log_dir / 'experiment_2b_call_tree.json'
log_file.write_text(json.dumps(result.metadata, indent=2))

print(f'Call tree saved to {log_file}')

# Load and display
loaded = json.loads(log_file.read_text())
print_tree(loaded['call_tree'])